# Design Decision Optimization with the JCSS LQI Criterion

This tutorial follows the continuous steel-bar decision example in [Schubert and Faber (2009)](https://www.jcss-lc.org/publications/raie/11_example_jcss_ms_2.pdf). The cross-sectional area changes the resistance, FORM estimates the failure probability, and the JCSS LQI criterion is compared with the economic optimum.

The design decision optimization helpers in Pystra are post-processing tools. They do not change the stochastic model or the reliability method; they use the `pf` and `beta` results returned by FORM, SORM, or simulation.

In [ ]:
import numpy as np
import pandas as pd
import pystra as ra

pd.options.display.float_format = "{:,.4g}".format

## Reliability model

The limit state is written as

$$g(f_y, A_s, S) = f_y A_s - 1000 S$$

where `A_s` is the design variable. For each value of `A_s`, FORM computes the failure probability and reliability index.

In [ ]:
def lsf(fy, As, S):
    return fy * As - 1000 * S


def run_reliability(As):
    limit_state = ra.LimitState(lsf)
    model = ra.StochasticModel()
    model.addVariable(ra.Lognormal("fy", 260, 18.2))
    model.addVariable(ra.Constant("As", As))
    model.addVariable(ra.Gumbel("S", 9.5, 1.5))

    options = ra.AnalysisOptions()
    options.setE1(1e-6)
    options.setE2(1e-6)

    form = ra.Form(
        analysis_options=options,
        limit_state=limit_state,
        stochastic_model=model,
    )
    form.run()
    return {"pf": float(np.atleast_1d(form.getFailure())[0]), "beta": form.getBeta()}

In [ ]:
as_values = np.array([70, 75, 80, 85.1, 90, 93, 95, 100], dtype=float)
study = ra.DesignStudy(variable="As", values=as_values, analysis=run_reliability)

reliability = study.evaluate()
reliability

## SWTP and the LQI target

Rackwitz's JCSS SWTP table is retained as a 1999 PPPUS$ anchor [Rackwitz (2008)](https://www.jcss-lc.org/publications/raie/06_risk_backgrounddoc_lqi_philosophy.pdf). For present-day studies, `indexed=True` returns an explicitly indexed value using World Bank GDP per capita PPP factors [World Bank WDI](https://data.worldbank.org/indicator/NY.GDP.PCAP.PP.CD). This keeps the literature value and the update method visible.

In [ ]:
swtp_anchor = ra.SWTP.from_country("CH")
swtp_indexed = ra.SWTP.from_country("CH", indexed=True)

pd.DataFrame(
    [
        {"basis": "Rackwitz anchor", "value_per_life": swtp_anchor.value_per_life, "price_year": swtp_anchor.price_year},
        {"basis": "GDP PPP indexed", "value_per_life": swtp_indexed.value_per_life, "price_year": swtp_indexed.price_year},
    ]
)

For the Fischer, Barnardo, and Faber target table [Fischer et al. (2012)](https://www.researchgate.net/publication/289533079_Deriving_target_reliabilities_from_the_LQI), define

$$K_1 = \frac{C_1}{\mathrm{SWTP} N_F}$$

where `C1` is the marginal safety cost and `N_F` is the expected number of fatalities conditional on failure. The result is a minimum LQI target reliability.

In [ ]:
expected_fatalities = 12
marginal_safety_cost = 5000

k1 = ra.lqi_k1(
    safety_cost_rate=marginal_safety_cost,
    swtp=swtp_indexed,
    expected_fatalities=expected_fatalities,
)
target = ra.lqi_target_reliability(k1)

pd.DataFrame([target.__dict__])

## Cost-benefit objective and LQI feasibility

The objective below follows the JCSS steel-bar calculation [Schubert and Faber (2009)](https://www.jcss-lc.org/publications/raie/11_example_jcss_ms_2.pdf) with a constant annual benefit, construction cost proportional to `A_s`, and failure costs discounted over the service life.

In [ ]:
costs = ra.CostBenefitModel(
    benefit_rate=1.2e4,
    interest_rate=0.02,
    service_life=100,
    construction_cost=lambda As: 5000 * As,
    failure_cost=lambda As: 5000 * As + 12 * 1.8e6 + 3e4,
)

assessment = ra.LQIAssessment(
    study=study,
    costs=costs,
    swtp=swtp_indexed,
    consequence=ra.FatalityConsequence(people_exposed=expected_fatalities),
    target=target,
)

results = assessment.evaluate()
results["construction_cost"] = 5000 * results["As"]
results["jcss_lqi_risk_cost"] = ra.jcss_lqi_risk_cost(
    results["construction_cost"],
    results["pf"],
    swtp_indexed,
    expected_fatalities,
)

results[[
    "As",
    "pf",
    "beta",
    "objective",
    "annualized_safety_cost",
    "target_pf",
    "lqi_acceptable",
    "jcss_lqi_risk_cost",
]]

In [ ]:
economic_best = results.loc[results["objective"].idxmax()]
feasible_best = results.loc[results[results["lqi_acceptable"]]["objective"].idxmax()]

pd.DataFrame(
    [
        {"selection": "economic optimum", "As": economic_best["As"], "pf": economic_best["pf"], "objective": economic_best["objective"], "lqi_acceptable": economic_best["lqi_acceptable"]},
        {"selection": "best LQI-feasible", "As": feasible_best["As"], "pf": feasible_best["pf"], "objective": feasible_best["objective"], "lqi_acceptable": feasible_best["lqi_acceptable"]},
    ]
)

In [ ]:
fig, axes = ra.plot_decision_summary(
    results,
    design="As",
    quantities=["objective", "pf"],
    labels={
        "As": r"cross section [mm$^2$]",
        "objective": "objective Z(A) [CHF]",
        "pf": r"failure probability $P_f(A)$ [yr$^{-1}$]",
    },
    reference_designs=[
        {"design": economic_best["As"], "label": "max objective", "marker": "o"},
        {"design": feasible_best["As"], "label": "best LQI-feasible", "marker": "s"},
    ],
    target_failure_probability=target.pf,
    figsize=(7.5, 5.0),
)
axes[0].set_title("Economic objective and LQI target")


## Schubert and Faber summary plot

Schubert and Faber's JCSS steel-bar example combines the failure probability, annualized safety cost, marginal LQI condition, and objective in one continuous-design summary. The next cell builds those quantities from the same reliability model and Table 1 parameters; the plotting call is intentionally generic, so it can be reused for other one-dimensional LQI design studies.

In [ ]:
from scipy.interpolate import PchipInterpolator
from scipy.optimize import brentq, minimize_scalar


jcss_designs = np.linspace(70.0, 100.0, 61)
jcss_curve = ra.DesignStudy("As", jcss_designs, run_reliability).evaluate()
jcss_pf = PchipInterpolator(jcss_curve["As"], jcss_curve["pf"])

paper_swtp = ra.SWTP.from_lqi(
    gross_domestic_product_per_capita=35931.0,
    mortality_rate=0.175,
    demographic_constant=18.9,
    currency="CHF",
    price_year=2009,
    source="Schubert and Faber (2009), Table 1",
)

publication_costs = ra.CostBenefitModel(
    benefit_rate=1.2e4,
    interest_rate=0.02,
    service_life=100,
    construction_cost=lambda As: 5000 * As,
    failure_cost=lambda As: 5000 * As + 12 * 1.8e6 + 3e4,
)


def pf_at(As):
    return float(jcss_pf(As))


def annualized_cost(As):
    return publication_costs.annualized_safety_cost(As, pf_at(As))


def objective_at(As):
    return publication_costs.objective(As, pf_at(As))


def marginal_lqi_term(As):
    dcost = ra.finite_difference_derivative(annualized_cost, As, step=1e-3)
    drate = ra.finite_difference_derivative(pf_at, As, step=1e-3)
    margin = ra.jcss_lqi_acceptability_margin(dcost, drate, paper_swtp, 12)
    return -margin / paper_swtp.for_lives(12)


optimal_as = float(
    minimize_scalar(lambda As: -objective_at(As), bounds=(70.0, 100.0), method="bounded").x
)
acceptable_as = float(brentq(marginal_lqi_term, 80.0, 100.0))

jcss_curve["annualized_safety_cost"] = [annualized_cost(As) / 1e3 for As in jcss_curve["As"]]
jcss_curve["lqi_marginal_term"] = [marginal_lqi_term(As) * 1e5 for As in jcss_curve["As"]]
jcss_curve["objective"] = [objective_at(As) for As in jcss_curve["As"]]

pd.DataFrame(
    [
        {
            "alternative": "economic optimum",
            "As": optimal_as,
            "reported_As": 85.1,
            "pf": pf_at(optimal_as),
            "annualized_safety_cost_10^3_CHF_per_year": annualized_cost(optimal_as) / 1e3,
            "objective_CHF": objective_at(optimal_as),
        },
        {
            "alternative": "LQI acceptance boundary",
            "As": acceptable_as,
            "reported_As": 93.0,
            "pf": pf_at(acceptable_as),
            "annualized_safety_cost_10^3_CHF_per_year": annualized_cost(acceptable_as) / 1e3,
            "objective_CHF": objective_at(acceptable_as),
        },
    ]
)


In [ ]:
fig, axes = ra.plot_decision_summary(
    jcss_curve,
    design="As",
    quantities=["pf", "annualized_safety_cost", "lqi_marginal_term", "objective"],
    labels={
        "As": r"cross section [mm$^2$]",
        "pf": r"$P_f(A)$ [yr$^{-1}$]",
        "annualized_safety_cost": r"$C_y(A)$ [$10^3$ CHF yr$^{-1}$]",
        "lqi_marginal_term": r"LQI marginal term [$10^{-5}$]",
        "objective": "objective function Z(A) [CHF]",
    },
    yscales={"pf": "log", "objective": "log"},
    invert_yaxis=["pf"],
    panel_labels=True,
    reference_designs=[
        {"design": optimal_as, "label": "economic optimum", "marker": "o"},
        {"design": acceptable_as, "label": "LQI boundary", "marker": "s"},
    ],
    figsize=(8.5, 8.5),
    line_kwargs={"marker": ""},
)
fig.suptitle("Schubert and Faber JCSS steel-bar example", y=1.01)
